# 📊 Modelo de Avaliação de Desempenho — Equipe de Processamento de Imagem

**Versão:** 2.3 | **Formato de saída:** PDF A3 Paisagem  
**Autor:** André Luiz Magalhães de Oliveira

---

## Objetivo

Avaliar comparativamente o desempenho de colaboradores de uma equipe de processamento de imagem médica (com foco em endodontia), gerando um **score final de 0 a 100** por meio de uma função logística calibrada.

O modelo integra quatro dimensões de desempenho:

| Dimensão | Descrição |
|---|---|
| **Produtividade** | Volume de fluxos processados, normalizado de forma robusta |
| **Qualidade** | Retrabalho excedente ao esperado para a complexidade técnica |
| **Eficiência** | Tempo médio de execução ajustado pela complexidade dos exames |
| **Complexidade** | Perfil de endodontia processada, ponderado por número de dentes |

## Função Logística

```
x_core = α·Produtividade − β·Retrabalho − γ·Tempo + δ·Complexidade + K·k_z
Score   = logistic(sensibilidade · x_core) × 100 × fator_confiança
```

O **fator de confiança** penaliza scores de colaboradores com baixo volume de exames, reduzindo ruído estatístico.

## Saída

Relatório PDF A3 com ranking geral, rankings por dimensão, distribuições, análise de sensibilidade e recomendações gerenciais automáticas.

## 1. Importações e Configuração de Caminhos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from textwrap import fill

# Caminhos de entrada e saída — ajuste conforme seu ambiente
CAMINHO_CSV = r"desempenho_processadores.csv"
CAMINHO_PDF = r"relatorio_desempenho_v23.pdf"

## 2. Parâmetros do Modelo

Todos os pesos e sensibilidades são centralizados aqui para facilitar calibração e auditoria.

| Parâmetro | Valor | Descrição |
|---|---|---|
| ALPHA | 1.0 | Peso da produtividade |
| BETA | 0.8 | Peso do retrabalho |
| GAMMA | 0.6 | Peso do tempo |
| DELTA | 0.8 | Peso da complexidade (endo) |
| K_WEIGHT | 0.5 | Peso do fator k_z |
| SENSIBILIDADE | 0.9 | Inclinação da curva logística |
| FATOR_AJUSTE_TEMPO_ENDO | 0.8 | Redução do tempo esperado para exames de endo |
| RETRAB_EXPONENTE | 1.2 | Expoente da penalização de retrabalho (não-linear) |
| CONF_N0 | 30 | Volume mínimo para fator de confiança pleno |

In [ ]:
ALPHA = 1.0
BETA = 0.8
GAMMA = 0.6
DELTA = 0.8
K_WEIGHT = 0.5
SENSIBILIDADE = 0.9
PESO_ENDO_ATE_3 = 0.3
PESO_ENDO_4A6 = 0.6
PESO_ENDO_7MAIS = 1.0
FATOR_AJUSTE_TEMPO_ENDO = 0.8
RETRAB_EXPONENTE = 1.2
CONF_N0 = 30

## 3. Funções Auxiliares

- **`logistic(x)`**: função sigmoide que mapeia qualquer valor real para o intervalo (0, 1)
- **`robust_z(series)`**: normalização robusta usando mediana e IQR, resistente a outliers

In [ ]:
def logistic(x):
    """Função logística (sigmoide): mapeia x para (0, 1)."""
    return 1 / (1 + np.exp(-x))


def robust_z(series: pd.Series) -> pd.Series:
    """
    Normalização robusta baseada em mediana e IQR.
    Menos sensível a outliers do que a normalização z padrão (média/desvio).
    Se IQR == 0, usa 1 para evitar divisão por zero.
    """
    med = series.median()
    iqr = series.quantile(0.75) - series.quantile(0.25)
    if iqr == 0:
        iqr = 1
    return (series - med) / iqr

## 4. Carregamento e Limpeza dos Dados

O CSV de entrada deve conter as colunas:
`nome`, `fluxos`, `retrabalhos`, `tempo_medio_execucao`, `k`, `endo_ate_3`, `endo_4a6`, `endo_7mais`

In [ ]:
def carregar_e_limpar_dados(caminho_csv: str) -> pd.DataFrame:
    """
    Lê o CSV de entrada, normaliza colunas, remove registros inválidos
    e converte colunas numéricas. Valores ausentes são preenchidos com 0.
    """
    df = pd.read_csv(caminho_csv, sep=";", decimal=",", encoding="utf-8")
    df.columns = df.columns.str.strip()

    df["nome"] = df["nome"].astype(str).str.strip()
    df = df[(df["nome"] != "") & (df["nome"] != "0") & (df["nome"].str.lower() != "nan")]

    colunas_numericas = [
        "fluxos", "retrabalhos", "tempo_medio_execucao",
        "k", "endo_ate_3", "endo_4a6", "endo_7mais"
    ]
    for col in colunas_numericas:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.fillna(0)
    return df

## 5. Pipeline de Cálculo do Score

O pipeline segue 4 etapas sequenciais:

1. **Complexidade de endodontia** — pondera os exames por dificuldade e normaliza
2. **Ajustes operacionais** — normaliza fluxo, tempo e retrabalho com `robust_z`
3. **Sub-scores** — transforma cada dimensão em score 0–100 via logística
4. **Score final** — combina sub-scores com a função logística e aplica fator de confiança

In [ ]:
def calcular_complexidade_endo(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula o score de complexidade de endodontia ponderando exames
    por número de dentes tratados. A raiz quadrada suaviza a escala.
    """
    df["endo_score_bruto"] = (
        df["endo_ate_3"] * PESO_ENDO_ATE_3 +
        df["endo_4a6"] * PESO_ENDO_4A6 +
        df["endo_7mais"] * PESO_ENDO_7MAIS
    )
    max_endo = df["endo_score_bruto"].max()
    df["endo_score_norm"] = df["endo_score_bruto"] / max_endo if max_endo > 0 else 0
    df["endo_eff"] = np.sqrt(df["endo_score_norm"])
    return df


def aplicar_ajustes_operacionais(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aplica ajustes no tempo (descontando complexidade de endo),
    normaliza fluxo e tempo com robust_z, e calcula retrabalho
    excedente em relação ao esperado para a complexidade.
    """
    # Tempo ajustado: colaboradores com endo complexa têm tolerância maior
    df["tempo_ajustado"] = df["tempo_medio_execucao"] / (
        1 + FATOR_AJUSTE_TEMPO_ENDO * df["endo_score_norm"]
    )
    df["fluxo_z"] = robust_z(df["fluxos"])
    df["tempo_z"] = robust_z(df["tempo_ajustado"])

    # Retrabalho esperado cresce com a complexidade; apenas o excedente é penalizado
    df["retrabalho_esperado"] = 1 + 0.8 * df["endo_score_norm"]
    df["retrabalho_excedente"] = (df["retrabalhos"] - df["retrabalho_esperado"]).clip(lower=0)
    df["retrab_z"] = robust_z(df["retrabalho_excedente"])

    # Saturação do fluxo e penalização não-linear do retrabalho
    df["fluxo_sat"] = np.tanh(df["fluxo_z"])
    df["retrab_penal"] = np.sign(df["retrab_z"]) * (np.abs(df["retrab_z"]) ** RETRAB_EXPONENTE)
    df["k_z"] = robust_z(df["k"])
    return df


def calcular_subscores(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula os quatro sub-scores (0–100) e um índice de equilíbrio
    entre dimensões (quanto mais próximo de 1, mais equilibrado).
    """
    df["sub_prod"] = logistic(df["fluxo_z"]) * 100
    df["sub_qual"] = logistic(-df["retrab_penal"]) * 100
    df["sub_efic"] = logistic(-df["tempo_z"]) * 100
    df["sub_complex"] = df["endo_eff"] * 100

    subs = df[["sub_prod", "sub_qual", "sub_efic", "sub_complex"]]
    df["sub_equilibrio"] = 1 - subs.std(axis=1) / (subs.std(axis=1).max() or 1)
    return df


def desempenho_logistico(fluxo_z, retrab_penal, tempo_z, k_z, endo_eff):
    """
    Combina os componentes na função logística central do modelo.
    Retorna um valor entre 0 e 1 (multiplicado por 100 no score final).
    """
    x_core = (
        ALPHA * fluxo_z
        - BETA * retrab_penal
        - GAMMA * tempo_z
        + DELTA * endo_eff
        + K_WEIGHT * k_z
    )
    return logistic(SENSIBILIDADE * x_core)


def calcular_score_final(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aplica a função logística e o fator de confiança baseado em volume.
    Colaboradores com baixo volume têm score penalizado proporcionalmente.
    Retorna o DataFrame ordenado por score ajustado (decrescente).
    """
    df["score_desempenho_bruto"] = df.apply(
        lambda r: desempenho_logistico(
            r["fluxo_z"], r["retrab_penal"], r["tempo_z"], r["k_z"], r["endo_eff"]
        ),
        axis=1
    ) * 100

    # Fator de confiança: cresce com o volume de exames até o limite CONF_N0
    df["conf_fator"] = np.sqrt(df["fluxos"] / (df["fluxos"] + CONF_N0)).clip(0, 1)
    df["score_desempenho_ajustado"] = df["score_desempenho_bruto"] * df["conf_fator"]

    return df.sort_values("score_desempenho_ajustado", ascending=False).reset_index(drop=True)

## 6. Análise de Sensibilidade

Avalia o impacto de variar cada parâmetro (ALPHA, BETA, GAMMA, DELTA) em ±20%,
medindo a variação média nos scores individuais. Identifica quais pesos têm maior
influência no resultado final.

In [ ]:
def analise_sensibilidade(df: pd.DataFrame) -> pd.DataFrame:
    """
    Para cada parâmetro principal, recalcula os scores com fator 0.8x e 1.2x
    e mede a variação média absoluta em relação ao score base.
    """
    base_scores = df["score_desempenho_ajustado"].values
    params = {"ALPHA": ALPHA, "BETA": BETA, "GAMMA": GAMMA, "DELTA": DELTA}
    resultados = []

    for nome, valor in params.items():
        for f in [0.8, 1.2]:
            if nome == "ALPHA":
                x_core_alt = (valor*f)*df["fluxo_z"] - BETA*df["retrab_penal"] - GAMMA*df["tempo_z"] + DELTA*df["endo_eff"] + K_WEIGHT*df["k_z"]
            elif nome == "BETA":
                x_core_alt = ALPHA*df["fluxo_z"] - (valor*f)*df["retrab_penal"] - GAMMA*df["tempo_z"] + DELTA*df["endo_eff"] + K_WEIGHT*df["k_z"]
            elif nome == "GAMMA":
                x_core_alt = ALPHA*df["fluxo_z"] - BETA*df["retrab_penal"] - (valor*f)*df["tempo_z"] + DELTA*df["endo_eff"] + K_WEIGHT*df["k_z"]
            else:
                x_core_alt = ALPHA*df["fluxo_z"] - BETA*df["retrab_penal"] - GAMMA*df["tempo_z"] + (valor*f)*df["endo_eff"] + K_WEIGHT*df["k_z"]

            score_alt = logistic(SENSIBILIDADE * x_core_alt) * 100 * df["conf_fator"]
            resultados.append({
                "Parâmetro": nome,
                "Fator": f,
                "Variação média (pontos)": round(np.mean(np.abs(score_alt - base_scores)), 2)
            })

    return pd.DataFrame(resultados)

## 7. Geração do Relatório PDF

O relatório é gerado em formato **A3 paisagem** com 8 páginas:

1. Capa e apresentação do modelo
2. Metodologia e sub-scores
3. Parâmetros e pesos de endodontia
4. Resumo estatístico e ranking Top 10
5. Distribuições dos scores
6. Rankings por dimensão (Top 5 cada)
7. Gráficos de ranking (geral e processadores com endo)
8. Análise de sensibilidade e recomendações gerenciais

In [ ]:
def criar_tabela_resumo(df):
    return pd.DataFrame({
        "Indicador": ["Total de colaboradores", "Score médio (ajustado)", "Score mediano (ajustado)",
                      "Score mínimo (ajustado)", "Score máximo (ajustado)", "Desvio padrão (ajustado)"],
        "Valor": [len(df), round(df["score_desempenho_ajustado"].mean(), 1),
                  round(df["score_desempenho_ajustado"].median(), 1),
                  round(df["score_desempenho_ajustado"].min(), 1),
                  round(df["score_desempenho_ajustado"].max(), 1),
                  round(df["score_desempenho_ajustado"].std(), 1)]
    })

def criar_tabela_top10(df, coluna_score, titulo_coluna):
    tabela = df[["nome", coluna_score]].copy()
    tabela = tabela.sort_values(coluna_score, ascending=False).head(10).reset_index(drop=True)
    tabela.insert(0, "Posição", range(1, len(tabela) + 1))
    tabela[coluna_score] = tabela[coluna_score].round(1)
    return tabela.rename(columns={coluna_score: titulo_coluna})

def criar_tabela_pesos_endo():
    return pd.DataFrame({
        "Tipo de Endodontia": ["Até 3 dentes", "4 a 6 dentes", "7 dentes ou mais"],
        "Peso aplicado no modelo": [PESO_ENDO_ATE_3, PESO_ENDO_4A6, PESO_ENDO_7MAIS]
    })

def criar_tabela_parametros_modelo():
    return pd.DataFrame({
        "Parâmetro": ["ALPHA (produtividade)", "BETA (retrabalho)", "GAMMA (tempo)",
                      "DELTA (complexidade)", "K_WEIGHT (peso de k_z)", "SENSIBILIDADE logística",
                      "FATOR_AJUSTE_TEMPO_ENDO", "RETRAB_EXPONENTE", "CONF_N0 (fator confiança)"],
        "Valor": [ALPHA, BETA, GAMMA, DELTA, K_WEIGHT, SENSIBILIDADE,
                  FATOR_AJUSTE_TEMPO_ENDO, RETRAB_EXPONENTE, CONF_N0]
    })

def pagina_texto(pdf, titulo, paragrafos, fontsize_titulo=18, fontsize_texto=12):
    fig, ax = plt.subplots(figsize=(16.53, 11.69))
    ax.axis("off")
    ax.text(0.5, 0.95, titulo, ha="center", va="top", fontsize=fontsize_titulo, fontweight="bold")
    y = 0.88
    for p in paragrafos:
        for linha in fill(p, width=140).split("\n"):
            ax.text(0.04, y, linha, ha="left", va="top", fontsize=fontsize_texto)
            y -= 0.03
        y -= 0.01
    pdf.savefig(fig)
    plt.close()

def pagina_tabelas(pdf, titulo, tabelas, bbox_list):
    fig, ax = plt.subplots(figsize=(16.53, 11.69))
    ax.axis("off")
    ax.text(0.5, 0.95, titulo, ha="center", va="top", fontsize=18, fontweight="bold")
    for (df_tab, subtitulo), bbox in zip(tabelas, bbox_list):
        if subtitulo:
            ax.text(bbox[0] + bbox[2]/2, bbox[1] + bbox[3] + 0.02, subtitulo,
                    ha="center", va="bottom", fontsize=14, fontweight="bold")
        t = ax.table(cellText=df_tab.values, colLabels=df_tab.columns, cellLoc="center", bbox=bbox)
        t.auto_set_font_size(False)
        t.set_fontsize(12)
        t.scale(1.1, 1.2)
    pdf.savefig(fig)
    plt.close()

def pagina_resumo_e_ranking(pdf, resumo, ranking):
    fig, ax = plt.subplots(figsize=(16.53, 11.69))
    ax.axis("off")
    ax.text(0.5, 0.95, "RESUMO E RANKING GERAL", ha="center", va="top", fontsize=18, fontweight="bold")
    ax.table(cellText=resumo.values, colLabels=resumo.columns, cellLoc="center", bbox=[0.08, 0.65, 0.84, 0.22])
    ax.text(0.5, 0.58, "Ranking — Top 10 (Score Ajustado)", ha="center", va="top", fontsize=14, fontweight="bold")
    t = ax.table(cellText=ranking.values, colLabels=ranking.columns, cellLoc="center", bbox=[0.02, 0.10, 0.96, 0.44])
    t.auto_set_font_size(False)
    t.set_fontsize(11)
    t.scale(1.05, 1.2)
    pdf.savefig(fig)
    plt.close()

def pagina_distribuicoes(pdf, df):
    fig, axes = plt.subplots(2, 2, figsize=(16.53, 11.69))
    fig.suptitle("Distribuições de Scores", fontsize=18, fontweight="bold")
    axes[0,0].hist(df["score_desempenho_ajustado"], bins=10, color="#4C72B0")
    axes[0,0].set_title("Distribuição do Score de Desempenho (Ajustado)")
    axes[0,0].grid(True, alpha=0.3)
    subs = df[["sub_prod", "sub_qual", "sub_efic", "sub_complex"]]
    axes[0,1].boxplot(subs.values, labels=["Prod", "Qual", "Efic", "Compl"])
    axes[0,1].set_title("Boxplot dos Sub-scores")
    axes[0,1].grid(True, axis="y", alpha=0.3)
    endo_df = df[df["endo_score_bruto"] > 0]
    axes[1,0].hist(endo_df["endo_score_bruto"], bins=10, color="#55A868")
    axes[1,0].set_title("Distribuição da Complexidade Endo")
    axes[1,0].grid(True, alpha=0.3)
    axes[1,1].scatter(df["fluxo_z"], df["retrab_penal"], alpha=0.7, color="#C44E52")
    axes[1,1].set_title("Fluxo (z) x Retrabalho Penalizado")
    axes[1,1].grid(True, alpha=0.3)
    fig.tight_layout(rect=[0, 0.02, 1, 0.95])
    pdf.savefig(fig)
    plt.close()

def pagina_rankings_subscores(pdf, df):
    df_tmp = df.copy()
    df_tmp["nome"] = df_tmp["nome"].apply(lambda x: fill(x, width=32))
    tabelas = [
        (criar_tabela_top10(df_tmp, "sub_prod", "Produtividade").head(5), "Top 5 — Produtividade"),
        (criar_tabela_top10(df_tmp, "sub_qual", "Qualidade").head(5), "Top 5 — Qualidade"),
        (criar_tabela_top10(df_tmp, "sub_efic", "Eficiência").head(5), "Top 5 — Eficiência"),
        (criar_tabela_top10(df_tmp, "sub_complex", "Complexidade").head(5), "Top 5 — Complexidade")
    ]
    bbox_list = [[0.05,0.55,0.40,0.30],[0.55,0.55,0.40,0.30],[0.05,0.10,0.40,0.30],[0.55,0.10,0.40,0.30]]
    fig, ax = plt.subplots(figsize=(16.53, 11.69))
    ax.axis("off")
    ax.text(0.5, 0.95, "RANKINGS POR DIMENSÃO", ha="center", va="top", fontsize=18, fontweight="bold")
    for (df_tab, subtitulo), bbox in zip(tabelas, bbox_list):
        ax.text(bbox[0]+bbox[2]/2, bbox[1]+bbox[3]+0.02, subtitulo, ha="center", va="bottom", fontsize=14, fontweight="bold")
        t = ax.table(cellText=df_tab.values, colLabels=df_tab.columns, cellLoc="center", bbox=bbox, colWidths=[0.26,0.96,0.40])
        t.auto_set_font_size(False)
        t.set_fontsize(12)
        t.scale(1.2, 1.4)
    pdf.savefig(fig)
    plt.close()

def pagina_graficos_rankings(pdf, df):
    fig, axes = plt.subplots(1, 2, figsize=(16.53, 11.69))
    fig.suptitle("Rankings Visuais", fontsize=18, fontweight="bold")
    axes[0].barh(df["nome"], df["score_desempenho_ajustado"], color="#4C72B0")
    axes[0].invert_yaxis()
    axes[0].set_title("Ranking Geral de Desempenho (Ajustado)")
    axes[0].set_xlabel("Score (0–100)")
    axes[0].grid(True, axis="x", alpha=0.3)
    endo_df = df[df["endo_score_bruto"] > 0].sort_values("score_desempenho_ajustado", ascending=False)
    axes[1].barh(endo_df["nome"], endo_df["score_desempenho_ajustado"], color="#55A868")
    axes[1].invert_yaxis()
    axes[1].set_title("Ranking — Processadores com Endodontia")
    axes[1].set_xlabel("Score (0–100)")
    axes[1].grid(True, axis="x", alpha=0.3)
    fig.tight_layout(rect=[0, 0.02, 1, 0.95])
    pdf.savefig(fig)
    plt.close()

def pagina_sensibilidade_e_recomendacoes(pdf, df_sens, df):
    fig, ax = plt.subplots(figsize=(16.53, 11.69))
    ax.axis("off")
    ax.text(0.5, 0.95, "ANÁLISE DE SENSIBILIDADE E RECOMENDAÇÕES", ha="center", va="top", fontsize=18, fontweight="bold")
    ax.text(0.5, 0.88, "Análise de sensibilidade dos parâmetros do modelo", ha="center", va="top", fontsize=14, fontweight="bold")
    ax.table(cellText=df_sens.values, colLabels=df_sens.columns, cellLoc="center", bbox=[0.05, 0.55, 0.90, 0.25])
    y = 0.48
    ax.text(0.05, y, "Recomendações gerenciais automáticas (baseadas nos dados atuais):",
            ha="left", va="top", fontsize=14, fontweight="bold")
    y -= 0.04
    textos = []
    crit = df[(df["sub_prod"] > df["sub_prod"].median()) & (df["sub_qual"] < df["sub_qual"].median())]["nome"].tolist()[:5]
    if crit:
        textos.append(f"- Colaboradores com alta produtividade e qualidade abaixo da mediana: {', '.join(crit)}. Recomenda-se análise das causas de retrabalho.")
    ref = df[(df["sub_complex"] > df["sub_complex"].median()) & (df["sub_efic"] > df["sub_efic"].median())]["nome"].tolist()[:5]
    if ref:
        textos.append(f"- Colaboradores com alta complexidade e boa eficiência: {', '.join(ref)}. Indicados como referência em treinamentos.")
    textos.append(f"- Índice médio de equilíbrio entre dimensões: {df['sub_equilibrio'].mean():.2f} (próximo de 1 = mais equilibrado).")
    if not df_sens.empty:
        s = df_sens.sort_values("Variação média (pontos)", ascending=False).iloc[0]
        textos.append(f"- Parâmetro com maior impacto: {s['Parâmetro']} ({s['Variação média (pontos)']} pontos de variação média com ajuste de ±20%).")
    for t in textos:
        for linha in fill(t, width=150).split("\n"):
            ax.text(0.05, y, linha, ha="left", va="top", fontsize=12)
            y -= 0.03
        y -= 0.01
    pdf.savefig(fig)
    plt.close()

def gerar_relatorio_pdf(caminho_pdf, df):
    """Orquestra a geração de todas as páginas do relatório PDF."""
    resumo = criar_tabela_resumo(df)
    top10 = criar_tabela_top10(df, "score_desempenho_ajustado", "Score Ajustado")
    pesos_endo = criar_tabela_pesos_endo()
    params = criar_tabela_parametros_modelo()
    sens = analise_sensibilidade(df)

    with PdfPages(caminho_pdf) as pdf:
        pagina_texto(pdf, "RELATÓRIO DE AVALIAÇÃO DE DESEMPENHO — VERSÃO 2.3 (A3)", [
            "Este relatório apresenta um modelo quantitativo, estruturado e auditável, destinado à avaliação comparativa do desempenho da equipe de processamento de imagem.",
            "A metodologia adota uma função logística com múltiplos componentes, integrando produtividade, qualidade técnica, eficiência operacional e complexidade dos exames processados.",
            "O modelo incorpora ajustes específicos para endodontia, normalizações robustas a outliers e um fator de confiança baseado em volume de exames.",
            "O score final varia de 0 a 100 e deve ser interpretado de forma comparativa entre colaboradores, sem caráter punitivo, servindo como instrumento de gestão e desenvolvimento."
        ])
        pagina_texto(pdf, "METODOLOGIA DO MODELO E SUB-SCORES", [
            "O modelo utiliza quatro sub-scores principais: Produtividade, Qualidade, Eficiência e Complexidade.",
            "Produtividade é baseada no volume processado (fluxos) normalizado de forma robusta.",
            "Qualidade considera apenas o retrabalho excedente ao esperado para a complexidade técnica.",
            "Eficiência avalia o tempo médio de execução ajustado pela complexidade dos exames.",
            "Complexidade reflete o perfil de endodontia processada, ponderando exames com diferentes números de dentes.",
            "Esses componentes são combinados em uma função logística calibrada, resultando em um score final único por colaborador."
        ])
        pagina_tabelas(pdf, "PARÂMETROS DO MODELO E PESOS DE ENDODONTIA",
            [(params, "Parâmetros gerais do modelo"), (pesos_endo, "Pesos aplicados à endodontia")],
            [[0.05, 0.55, 0.90, 0.30], [0.20, 0.15, 0.60, 0.25]])
        pagina_resumo_e_ranking(pdf, resumo, top10)
        pagina_distribuicoes(pdf, df)
        pagina_rankings_subscores(pdf, df)
        pagina_graficos_rankings(pdf, df)
        pagina_sensibilidade_e_recomendacoes(pdf, sens, df)

## 8. Execução do Pipeline

Rode a célula abaixo para executar o pipeline completo e gerar o relatório PDF.

In [ ]:
def main():
    df = carregar_e_limpar_dados(CAMINHO_CSV)
    df = calcular_complexidade_endo(df)
    df = aplicar_ajustes_operacionais(df)
    df = calcular_subscores(df)
    df = calcular_score_final(df)
    gerar_relatorio_pdf(CAMINHO_PDF, df)
    print(f"Relatório gerado com sucesso em: {CAMINHO_PDF}")


main()